## Підбор параметра partitionBlock та filterBlock
Зафіксовані параметри під час підбору:
- `threshold` (базовий випадок в послідовному): 8192
- `parallelSortCutoff` (поріг, коли включається паралельне сортування): 0 — значить, що паралельного сортування не буде
- `parallelCutoff`: 131072

`partitionBlock` підбирався на параметрах 16384, 32768, 65536.

`filterBlock` підбирався на параметрах 4096, 8192, 16384, 32768.

Для вибору розмірів блоків застосовувався той самий критерій, що й для порогу паралельності. Найменше середнє геометричне відносного часу (1,011) мала комбінація з блоком розбиття 16384 і блоком фільтра 4096. Комбінації 16384, 8192 і 32768, 4096 відставали від неї менш ніж на 0,5%, що не перевищує розкиду (1.5%) між повторами. Великі блоки розбиття (65536) були помітно гіршими: середнє відставання 1,036–1,041, а на найгіршому графі - до 14,6%. Оскільки найкращі значення виявилися найменшими в досліджуваному діапазоні, було проведено додатковий прогін із меншими розмірами блоків.

### Обране значення partitionBlock: `16384`
### Обране значення filterBlock: `4096`

In [22]:
# Дані
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [23]:
df1 = pd.read_csv("C:/Users/admin/source/repos/Kruskal/results/fk_par_blocks.csv")
df2 = pd.read_csv("C:/Users/admin/source/repos/Kruskal/results/fk_par_cutoff.csv")
result = pd.concat([df2[df2["algorithm"] == "fk"], df1], ignore_index=True)
MIN_EDGES = 1_000_000          # поріг впливає лише на графи, де ребер помітно більше за нього

In [24]:
graphs = (
    result[["series", "graph", "V", "E"]]
    .drop_duplicates()
    .sort_values(["series", "V", "E"])
    .reset_index(drop=True)
)

display(graphs)

,series,graph,V,E
0,density,B1,8000,16000
1,density,B2,8000,32000
2,density,B3,8000,64000
3,density,B4,8000,128000
4,density,B5,8000,256000
5,density,B6,8000,512000
6,density,B7,8000,1024000
7,density,B8,8000,2048000
8,density,B9,8000,4096000
9,density,B10,8000,8192000


In [25]:
df = result.copy()

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4619 entries, 0 to 4618
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   series           4619 non-null   object 
 1   graph            4619 non-null   object 
 2   family           4619 non-null   object 
 3   algorithm        4619 non-null   object 
 4   threads          4619 non-null   int64  
 5   threshold        4619 non-null   int64  
 6   partition_block  4265 non-null   float64
 7   filter_block     4265 non-null   float64
 8   par_cutoff       4265 non-null   float64
 9   psort_cutoff     4265 non-null   float64
 10  V                4619 non-null   int64  
 11  E                4619 non-null   int64  
 12  E_over_V         4619 non-null   float64
 13  seed             4619 non-null   int64  
 14  repeat           4619 non-null   int64  
 15  total_ms         4619 non-null   float64
 16  sort_ms          0 non-null      float64
 17  dsu_ms        

In [27]:
# Параметри досліду
PARAMS = ["threads", "threshold", "partition_block", "filter_block", "par_cutoff", "psort_cutoff"]

def experiment_params(df):
    rows = {}
    for alg, d in df.groupby("algorithm", sort=False):
        row = {}
        for c in PARAMS:
            vals = sorted(d[c].dropna().unique())
            row[c] = ", ".join(str(int(v)) for v in vals) if vals else "-"
        repeats = d.groupby(["graph"] + PARAMS, dropna=False)["repeat"].nunique()
        row["повторів"] = f"{repeats.min()}–{repeats.max()}" if repeats.min() != repeats.max() else str(repeats.min())
        rows[alg] = row
    graphs = df["graph"].unique()
    print(f"Графи ({len(graphs)}): {', '.join(graphs)}")
    return pd.DataFrame(rows).T

display(experiment_params(df))

Графи (19): B1, B2, B3, B4, B5, B6, B7, B8, B9, B10, B11, G1, C1, C2, C3, C4, C5, C6, C7


,threads,threshold,partition_block,filter_block,par_cutoff,psort_cutoff,повторів
fk,1,8192,-,-,-,-,6–25
fk-par,12,8192,"16384, 32768, 65536","4096, 8192, 16384, 32768",131072,0,7–25


In [28]:
order = list(df["graph"].unique())   # порядок графів як у плані

In [29]:
# %% 1. Середній час: послідовний FK і fk-par для кожного порогу
seq = df[df["algorithm"] == "fk"].groupby("graph")["total_ms"].mean()

par = df[df["algorithm"] == "fk-par"].copy()
par["partition_block"] = par["partition_block"].astype(int)
par["filter_block"] = par["filter_block"].astype(int)
t = par.groupby(["graph", "E", "partition_block", "filter_block"], as_index=False)["total_ms"].mean()

t["speedup"] = t["graph"].map(seq) / t["total_ms"]                          # прискорення відносно послідовного FK
t["rel"] = t["total_ms"] / t.groupby("graph")["total_ms"].transform("min")   # 1.00 = найкраща комбінація блоків на графі

print("Середній час fk-par, мс:")
display(t.pivot(index="graph", columns=["partition_block", "filter_block"], values="total_ms").reindex(order).round(1))

Середній час fk-par, мс:


partition_block   16384                           32768                  \
filter_block      4096    8192    16384   32768   4096    8192    16384   
graph                                                                     
B1                  0.9     0.9     0.9     0.9     0.9     0.9     0.9   
B2                  1.1     1.1     1.2     1.1     1.0     1.0     1.0   
B3                  1.4     1.4     1.4     1.4     1.4     1.5     1.4   
B4                  2.3     2.2     2.3     2.3     2.3     2.2     2.3   
B5                  3.6     3.7     3.6     3.7     3.6     3.7     3.6   
B6                  5.5     5.5     5.5     5.5     5.4     5.4     5.4   
B7                  9.6     9.9    10.0    10.0    10.1    10.1    10.2   
B8                 18.4    18.2    18.5    18.8    18.9    19.0    18.5   
B9                 37.2    37.5    37.7    37.8    37.0    36.8    36.8   
B10                80.1    81.3    81.7    80.7    80.3    79.7    79.7   
B11               309.9   314.0   311.9   317.3   306.2   309.9   312.2   
G1                912.4   909.6   910.1   920.5   897.7   898.8   906.6   
C1                  1.8     1.7     1.8     1.7     1.7     1.8     1.7   
C2                  6.4     6.4     6.4     6.5     6.1     6.2     6.2   
C3                 18.3    18.1    18.5    19.0    18.2    17.7    18.4   
C4                 54.9    54.8    55.3    56.1    55.2    55.2    56.1   
C5                200.8   202.8   207.9   212.8   209.3   204.9   207.7   
C6                891.8   869.1   868.4   867.6   854.1   868.4   876.1   
C7               3591.1  3595.9  3622.7  3653.9  3601.2  3668.0  3616.5   

partition_block           65536                          
filter_block      32768   4096    8192    16384   32768  
graph                                                    
B1                  0.9     0.9     0.9     0.9     0.9  
B2                  1.0     1.0     1.0     1.0     1.0  
B3                  1.4     1.4     1.4     1.4     1.4  
B4                  2.2     2.3     2.3     2.3     2.3  
B5                  3.6     3.6     3.6     3.6     3.6  
B6                  5.5     5.8     5.8     5.8     5.8  
B7                 10.4    11.0    10.8    10.8    10.5  
B8                 18.8    19.5    19.4    19.5    19.5  
B9                 36.6    37.4    38.2    37.7    37.8  
B10                79.0    79.2    78.9    79.2    79.5  
B11               311.7   315.1   313.7   307.2   309.6  
G1                920.6   910.5   920.6   915.4   938.4  
C1                  1.8     1.8     1.8     1.8     1.8  
C2                  6.2     6.2     6.3     6.3     6.3  
C3                 18.8    18.5    18.8    18.7    18.9  
C4                 57.1    55.5    55.7    55.9    57.0  
C5                209.7   209.7   211.2   213.5   211.5  
C6                880.2   871.8   872.9   875.4   881.3  
C7               3638.1  3626.3  3636.4  3683.3  3700.9

In [30]:
print("Прискорення відносно послідовного FK:")
display(t.pivot(index="graph", columns=["partition_block", "filter_block"], values="speedup").reindex(order).round(2))

Прискорення відносно послідовного FK:


partition_block 16384                   32768                   65536        \
filter_block    4096  8192  16384 32768 4096  8192  16384 32768 4096  8192    
graph                                                                         
B1               0.96  0.99  0.99  0.99  1.01  0.99  1.01  1.00  0.98  1.00   
B2               1.01  0.97  0.94  0.96  1.06  1.08  1.07  1.05  1.06  1.07   
B3               1.16  1.12  1.16  1.13  1.15  1.07  1.12  1.13  1.12  1.10   
B4               1.04  1.05  1.04  1.04  1.04  1.05  1.04  1.05  1.03  1.03   
B5               0.97  0.95  0.97  0.95  0.97  0.96  0.96  0.97  0.97  0.98   
B6               1.21  1.20  1.19  1.20  1.21  1.23  1.22  1.20  1.14  1.14   
B7               1.27  1.24  1.23  1.22  1.21  1.21  1.19  1.18  1.11  1.13   
B8               1.38  1.40  1.38  1.35  1.35  1.34  1.37  1.36  1.31  1.31   
B9               1.29  1.28  1.28  1.27  1.30  1.31  1.30  1.31  1.28  1.26   
B10              1.24  1.22  1.21  1.23  1.23  1.24  1.24  1.25  1.25  1.25   
B11              1.25  1.23  1.24  1.22  1.26  1.25  1.24  1.24  1.23  1.23   
G1               1.43  1.43  1.43  1.41  1.45  1.45  1.44  1.41  1.43  1.41   
C1               0.98  1.01  1.01  1.01  1.01  1.00  1.01  1.01  1.01  1.01   
C2               0.94  0.93  0.93  0.92  0.98  0.98  0.97  0.97  0.97  0.96   
C3               1.16  1.17  1.15  1.12  1.17  1.20  1.16  1.13  1.15  1.13   
C4               1.26  1.26  1.25  1.23  1.25  1.25  1.23  1.21  1.24  1.24   
C5               1.29  1.27  1.24  1.21  1.24  1.26  1.24  1.23  1.23  1.22   
C6               1.49  1.53  1.53  1.53  1.56  1.53  1.52  1.51  1.53  1.52   
C7               1.62  1.62  1.61  1.59  1.62  1.59  1.61  1.60  1.61  1.60   

partition_block              
filter_block    16384 32768  
graph                        
B1               0.99  1.00  
B2               1.04  1.07  
B3               1.09  1.08  
B4               1.04  1.02  
B5               0.96  0.98  
B6               1.14  1.14  
B7               1.13  1.16  
B8               1.31  1.31  
B9               1.27  1.27  
B10              1.25  1.25  
B11              1.26  1.25  
G1               1.42  1.39  
C1               0.99  1.00  
C2               0.95  0.95  
C3               1.13  1.12  
C4               1.23  1.21  
C5               1.21  1.22  
C6               1.52  1.51  
C7               1.58  1.57

In [31]:
# %% 2. Висновок: лише великі графи
big = t[t["E"] >= MIN_EDGES]
print(f"Графи з E >= {MIN_EDGES:,}: {', '.join(g for g in order if g in set(big['graph']))}")

rel_table = big.pivot(index="graph", columns=["partition_block", "filter_block"], values="rel").reindex([g for g in order if g in set(big['graph'])])
print("Відносний час (1.00 = найкращий поріг на цьому графі):")
display(rel_table.round(3))

Графи з E >= 1,000,000: B7, B8, B9, B10, B11, G1, C4, C5, C6, C7
Відносний час (1.00 = найкращий поріг на цьому графі):


partition_block  16384                       32768                       \
filter_block     4096   8192   16384  32768  4096   8192   16384  32768   
graph                                                                     
B7               1.000  1.025  1.034  1.035  1.050  1.051  1.061  1.076   
B8               1.009  1.000  1.014  1.033  1.035  1.041  1.016  1.030   
B9               1.017  1.026  1.030  1.033  1.010  1.004  1.006  1.000   
B10              1.016  1.030  1.035  1.023  1.018  1.011  1.010  1.001   
B11              1.012  1.026  1.019  1.036  1.000  1.012  1.020  1.018   
G1               1.016  1.013  1.014  1.025  1.000  1.001  1.010  1.025   
C4               1.001  1.000  1.008  1.023  1.007  1.007  1.023  1.042   
C5               1.000  1.010  1.035  1.060  1.042  1.020  1.034  1.044   
C6               1.044  1.018  1.017  1.016  1.000  1.017  1.026  1.030   
C7               1.000  1.001  1.009  1.017  1.003  1.021  1.007  1.013   

partition_block  65536                       
filter_block     4096   8192   16384  32768  
graph                                        
B7               1.146  1.125  1.122  1.095  
B8               1.069  1.064  1.069  1.069  
B9               1.022  1.043  1.030  1.033  
B10              1.004  1.000  1.004  1.007  
B11              1.029  1.025  1.003  1.011  
G1               1.014  1.025  1.020  1.045  
C4               1.013  1.016  1.020  1.040  
C5               1.044  1.051  1.063  1.053  
C6               1.021  1.022  1.025  1.032  
C7               1.010  1.013  1.026  1.031

In [33]:
summary = (big.groupby(["partition_block", "filter_block"])["rel"]
              .agg(середнє_відставання=lambda r: np.exp(np.log(r).mean()),
                   найгірший_граф="max",
                   перемог=lambda r: int((r == 1.0).sum()))
              .sort_values("середнє_відставання"))
display(summary.round(3))

середнє_відставання  найгірший_граф  перемог
partition_block filter_block                                              
16384           4096                        1.011           1.044        3
                8192                        1.015           1.030        2
32768           4096                        1.016           1.050        3
                8192                        1.018           1.051        0
                16384                       1.021           1.061        0
16384           16384                       1.021           1.035        0
32768           32768                       1.028           1.076        1
16384           32768                       1.030           1.060        0
65536           4096                        1.036           1.146        0
                16384                       1.038           1.122        0
                8192                        1.038           1.125        1
                32768                       1.041           1.095        0

In [18]:
best = summary.index[0]
print(f"\nНайкращий розмір блоку: partition={best[0]}, filter={best[1]}")
print(f"Він повільніший за найкращий на кожному графі в середньому на {100 * (summary.iloc[0, 0] - 1):.1f}%")


Найкращий розмір блоку: partition=16384, filter=4096
Він повільніший за найкращий на кожному графі в середньому на 1.1%


In [25]:
noise = (par[par["E"] >= MIN_EDGES].groupby(["graph", "partition_block", "filter_block"])["total_ms"]
            .agg(lambda x: x.std() / x.mean()).median() * 100)
print(f"Типовий розкид між повторами: {noise:.1f}%  (різниця, менша за нього - це шум)")

Типовий розкид між повторами: 1.5%  (різниця, менша за нього — це шум)
